<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/labs/cs473_lab_week_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b>After clicking the "Open in Colab" link, copy the notebook to your own Google Drive before getting started, or it will not save your work</b></p>

# BYU CS 473 Lab Week 11

## Introduction:

In this lab you will be comparing different boosting techniques, specifically the AdaBoost and LogitBoost algorithms.

AdaBoost is a powerful ensemble learning algorithm designed to improve the performance of weak classifiers by combining them into a strong classifier. In AdaBoost, a sequence of simple models is trained iteratively, with each model focusing more on the instances that previous models misclassified. The algorithm works by assigning equal weights to all training samples at the beginning. After each iteration, the weights of misclassified samples are increased, causing subsequent base classifiers to focus on the harder cases. This adaptation helps the ensemble address both bias and variance, often resulting in a more accurate and robust model than any single weak classifier. AdaBoost aggregates the predictions of all weak learners using a weighted voting scheme, where each learner's contribution depends on its accuracy.

LogitBoost is another boosting algorithm for classification that builds an ensemble of weak learners to predict class probabilities accurately. Unlike AdaBoost, which focuses on minimizing exponential loss, LogitBoost is designed to minimize logistic loss, making it especially suitable for tasks where well-calibrated probability estimates are important. LogitBoost works by fitting each weak learner in a stage-wise fashion. At each boosting round, the algorithm calculates pseudo-residuals called "working responses" and corresponding weights based on the current model's probability estimates. The base learner then fits these responses using weighted least squares. Iteratively, this process refines the ensemble so it improves the accuracy of predicted probabilities. LogitBoost is helpful because it tends to be more robust to noisy data and outliers compared to AdaBoost, and it provides reliable probability outputs that can be used in decision-making or as input for other models. This makes LogitBoost a strong choice when you want both high accuracy and meaningful probability estimates in your classification models.

See Sections 18.5.3 and 18.5.4 for more details on these algorithms.

---
## Grading standards   

Your notebook will be graded on the following:

* 40% Correct implementation of the AdaBoost algorithm
* 40% Correct implementation of the LogitBoost algorithm
* 20% Analysis of the different methods
---

#### Imports and Data Preprocessing

In [27]:
from datasets import load_dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, classification_report

data = load_dataset("AiresPucrs/adult-census-income", split="train").to_pandas()

data.replace('?', pd.NA, inplace=True)
data = data.dropna()

# Data cleaning
categorials = data.columns[data.dtypes == object].values
encoders = {col:LabelEncoder() for col in categorials}
for col in categorials:
    data[col] = encoders[col].fit_transform(data[col])

target_col = data.columns[-1]

# Use this data for the single classifier and LogitBoost
X, y = data.iloc[:, :-1], data.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Change target labels to -1 and 1 instead of 0 and 1 to work with AdaBoost algorithm
# Use this data for AdaBoost
data[target_col] = data[target_col].replace({0: -1, 1: 1})
X, y = data.iloc[:, :-1], data.iloc[:, -1]
X_train_adaboost, X_test_adaboost, y_train_adaboost, y_test_adaboost = train_test_split(X, y, test_size=0.2, random_state=42)

### Single Classifier

Below is a single decision tree classifier that has been fit to the data. Run the cell to see how it does on this dataset.

In [28]:
clf = DecisionTreeClassifier(max_depth=1, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Single Decision Tree Classifier Accuracy: {accuracy:.4f}")

# Detailed classification report
print(classification_report(y_test, y_pred))

Single Decision Tree Classifier Accuracy: 0.7514
              precision    recall  f1-score   support

           0       0.75      1.00      0.86      4533
           1       0.00      0.00      0.00      1500

    accuracy                           0.75      6033
   macro avg       0.38      0.50      0.43      6033
weighted avg       0.56      0.75      0.64      6033



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Implement AdaBoost

Following Algorithm 18.1 on page 616 of the textbook, you will implement the AdaBoost algorithm. For the base classifier, use sklearn.DecisionTreeClassifier with max_depth=1 as used above for the single classifier example. After you finish your AdaBoost implementation, get the predictions of your model on X_test_adaboost and print the accuracy and classification report as done above for the single classifier.

![Adaboost Algorithm](https://raw.githubusercontent.com/wingated/cs473/main/labs/images/lab_week_11_adaboost.png)

Hints:

* AdaBoost relies on class labels being {+1, -1}. To handle this, use X_train_adaboost/y_train_adaboost which have been correctly labeled.

* w should be shape (N,) with one value per datapoint.

* When computing alpha, add a small epsilon term (1e-10) to the denominator for numerical stability.

* For each iteration, you should save your classifier and your alpha to use for the final prediction (last line of the pseudocode).

* The sgn[] in the last line means you should take the sign of the weighted sum inside the brackets.

In [29]:
import numpy as np

In [30]:
# Your AdaBoost Implementation Here


def AdaBoost(aX_train, ay_train):
    #1)
    w = np.ones(len(aX_train))/len(aX_train)
    #2)
    M = 50
    F_list = []
    aph_list = []
    for i  in range(M):
      #3)
      clf_i = DecisionTreeClassifier(max_depth=1, random_state=42)
      clf_i.fit(aX_train, ay_train, sample_weight=w)
      y_pred = clf_i.predict(aX_train)
      #4)
      err_m = np.sum(w*(ay_train!=y_pred))/np.sum(w)
      #5)
      eps = 1e-10
      aph = np.log((1-err_m)/(err_m + eps))
      #6
      w = w*np.exp(aph*(y_pred!=ay_train))

      # put into full vectors
      aph_list.append(aph)
      F_list.append(clf_i)

    return aph_list, F_list


    #to use:
def predict_adaboost(X, clfs, alphas):
    # Initialize running sum of weighted predictions
    final_preds = np.zeros(len(X))

    for clf, alpha in zip(clfs, alphas):
        final_preds += alpha * clf.predict(X)

    # Take the sign: +1 for >50k, -1 for <=50k
    return np.sign(final_preds)





In [31]:
# Use the predictions of your AdaBoost model to get accuracy from the test set
aph, F = AdaBoost(X_train_adaboost, y_train_adaboost)

In [32]:
#predictions:
y_pred_adaboost = predict_adaboost(X_test_adaboost, F, aph)

# Evaluate accuracy
accuracy = accuracy_score(y_test_adaboost, y_pred_adaboost)
print(f"AdaBoost Accuracy: {accuracy:.4f}")

# Detailed classification report
print(classification_report(y_test_adaboost, y_pred_adaboost))

AdaBoost Accuracy: 0.8419
              precision    recall  f1-score   support

          -1       0.87      0.93      0.90      4533
           1       0.73      0.57      0.64      1500

    accuracy                           0.84      6033
   macro avg       0.80      0.75      0.77      6033
weighted avg       0.83      0.84      0.84      6033



### Implement LogitBoost

Following Algorithm 18.2 on page 617 of the textbook, implement the LogitBoost algorithm. As your base learner, use sklearn.DecisionTreeRegressor with max_depth=1.

![LogitBoost Algorithm](https://raw.githubusercontent.com/wingated/cs473/main/labs/images/lab_week_11_logitboost.png)

Hints:

* LogitBoost uses class labels of {1, 0}. X_train/y_train should be formatted for this.

* w will again be shape (N,), as will pi.

* Similar to alpha in AdaBoost, when computing z, add a small epsilon term (1e-10) to the denominator for numerical stability.

* Calculating $F_{m}$ will be as simple as using the DecisionTreeRegressor's fit function (using X and z for your data, and w for the sample_weight).

* $f(x)$ will initially be all zeros of shape (N,).

* Again, save each of your regressors for the final prediction step.

In [33]:
# Your code here

def wrkresp(pi, y):
  '''takes  in vector of pi's and vector of true labels and
  returns vector of working responses'''

  w = pi*(1-pi)
  w = np.maximum(w, 1e-10)

  z = (y-pi)/w

  return z, w




In [34]:
def Logitboost(X, y):
  '''takes  in vector of pi's and vector of true labels and
  returns vector of working responses'''
  N = len(y)
  M = 50
  F = np.zeros(N)
  regressors = []
  for i in range(M):
          # get probs
          pi = 1 / (1 + np.exp(-F))

          # Get working response and w's
          z, w = wrkresp(pi, y)

          # Fit
          model = DecisionTreeRegressor(max_depth=1, random_state=42)
          model.fit(X, z, sample_weight=w)

          # Update F(x)
          F += 0.5 * model.predict(X)
          regressors.append(model)

  return regressors

  #  prediction function
def predict_logitboost(X, regressors):
    '''Uses the trained LogitBoost ensemble to make predictions on new data.'''
    #initialize storage
    F_test = np.zeros(len(X))

    # Pass the data through every tree we saved
    for model in regressors:
        F_test += 0.5 * model.predict(X)
    #final score preduction
    predictions = (F_test > 0).astype(int)
    return predictions

In [35]:
ex_pi = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
ex_y = np.array([1, 0, 1, 0, 1])
wrkresp(ex_pi, ex_y)

(array([10.        , -1.25      ,  3.33333333, -1.66666667,  2.        ]),
 array([0.09, 0.16, 0.21, 0.24, 0.25]))

In [36]:
regressors = Logitboost(X_train, y_train)
y_pred_l = predict_logitboost(X_test, regressors)

In [37]:
# Use the predictions of your LogitBoost model to get accuracy from the test set

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred_l)
print(f"Logitboost Accuracy: {accuracy:.4f}")

# Detailed classification report
print(classification_report(y_test, y_pred_l))


Logitboost Accuracy: 0.8492
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      4533
           1       0.77      0.56      0.65      1500

    accuracy                           0.85      6033
   macro avg       0.82      0.75      0.78      6033
weighted avg       0.84      0.85      0.84      6033



### Analysis

Compare the accuracy of each model (single decision tree classifier, AdaBoost, and LogitBoost) and describe what you learned about the effectiveness of boosting methods (2-3 sentences).

#Accuracy Analysis:
We can see that the single decision tree entirely failed to use any features. It only predicted 0 for every single datapoint, resulting in a precision, recall and f1-score of 0 for class (1) and an overall accuracy equal to the class proportions, .7514.

Adaboost got around this issue but struggled with recall on class (1) (it could only label correctly 57% of points in class 1)
Logitboost is strikingly similar in summary statistics, almost exactly identical, but noticeably better precision of class 1 and therefore recall of class 0 than that of adaboost's.

#What I Learned:

I learned that the ability to boost allows us to stop our models from taking the easy (dumb) way out. As we ensembled, our accuracy skyrocketed from a fake 75% to a real 84-85%. The probabilistic logitboost also did slightly better, which makes sense as it deals with soft boundaries better.

# Big Table:

Single Decision Tree Classifier Accuracy: 0.7514
              
                  precision    recall  f1-score   support

           0       0.75      1.00      0.86      4533
           1       0.00      0.00      0.00      1500

    accuracy                           0.75      6033
    macro avg       0.38      0.50      0.43      6033
    weighted avg    0.56      0.75      0.64      6033


AdaBoost Accuracy: 0.8419
              
                precision    recall  f1-score   support

          -1       0.87      0.93      0.90      4533
           1       0.73      0.57      0.64      1500

    accuracy                           0.84      6033
    macro avg       0.80      0.75      0.77      6033
    weighted avg    0.83      0.84      0.84      6033


Logitboost Accuracy: 0.8492
              
                  precision    recall  f1-score   support

           0       0.87      0.94      0.90      4533
           1       0.77      0.56      0.65      1500

    accuracy                           0.85      6033
    macro avg       0.82      0.75      0.78      6033
    weighted avg    0.84      0.85      0.84      6033